In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd

from diaphanous.show import show

# Rebuild linear model for decade 2014-2023
reports = (
    pd.read_csv(
        '../data/ocse-reports-per-year.csv',
        thousands=',',
        index_col='year'
    )
    .sort_index(ascending=True)
)

decade = reports[(2014 <= reports.index) & (reports.index <= 2023)]
fit = np.polyfit(decade.index, decade["reports"], 1)

# Determine informational v actionable reports
ia = pd.read_csv("../data/ocse-report-recipients.csv", thousands=",")
ia = (
    ia
    .groupby(["year", "level_of_detail"], observed=False)
    .sum()
    .drop(columns="recipients")
    .reset_index(level=1)
    .pivot(columns="level_of_detail", values="reports")
    .assign(reports=lambda df: df["informational"] + df["actionable"])
    .assign(act_pct=lambda df: df["actionable"] / df["reports"] * 100)
    .assign(info_pct=lambda df: df["informational"] / df["reports"] * 100)
)

show(ia, caption="Informational v Actionable Reports")

# Assemble all necessary metrics
metrics = pd.DataFrame({
    "description": [
        "linear model: yearly growth",
        "linear model: y-axis intercept",
        "mean fraction of actionable reports 2020–2023",
        "mean fraction of actionable reports 2020, 2022, 2023",
        "fraction of actionable reports 2024",
        "mean fraction of informational reports 2020–2023",
        "mean fraction of informational reports 2020, 2022, 2023",
        "fraction of informational reports 2024",
        "reports in 2023",
        "reports in 2024",
        "incidents in 2024",
    ],
    "value": [
        fit[0],
        fit[1],
        ia.loc[(ia.index != 2024), "act_pct"].mean(),
        ia.loc[(ia.index != 2024) & (ia.index != 2021), "act_pct"].mean(),
        ia.loc[2024, "act_pct"],
        ia.loc[(ia.index != 2024), "info_pct"].mean(),
        ia.loc[(ia.index != 2024) & (ia.index != 2021), "info_pct"].mean(),
        ia.loc[2024, "info_pct"],
        reports.loc[2023, "reports"],
        reports.loc[2024, "reports"],
        29_200_000,
    ]
}, index=[
    "growth",
    "constant",
    "act_pct_old0",
    "act_pct_old",
    "act_pct_new",
    "info_pct_old0",
    "info_pct_old",
    "info_pct_new",
    "r2023",
    "r2024",
    "i2024",
])

show(metrics, caption="Collected Metrics")

def metric_value(name):
    return metrics.loc[name, "value"]

growth = metric_value("growth")
act_old_prime = metric_value("act_pct_old") / 100
act_new_prime = metric_value("act_pct_new") / 100
info_old = metric_value("info_pct_old") / 100
info_new = metric_value("info_pct_new") / 100
r2023 = metric_value("r2023")
r2024 = metric_value("r2024")
i2024 = metric_value("i2024")

level_of_detail,actionable,informational,reports,act_pct,info_pct
year,,,,,
2020,"10,048,818","11,732,463","21,781,281",46.1,53.9
2021,"18,442,566","11,002,617","29,445,183",62.6,37.4
2022,"15,845,672","16,408,606","32,254,278",49.1,50.9
2023,"18,358,338","18,163,560","36,521,898",50.3,49.7
2024,"16,612,392","4,566,847","21,179,239",78.4,21.6


,description,value
growth,linear model: yearly growth,"3,928,236.4"
constant,linear model: y-axis intercept,"-7,911,256,089.5"
act_pct_old0,mean fraction of actionable reports 2020–2023,52.0
act_pct_old,"mean fraction of actionable reports 2020, 2022, 2023",48.5
act_pct_new,fraction of actionable reports 2024,78.4
info_pct_old0,mean fraction of informational reports 2020–2023,48.0
info_pct_old,"mean fraction of informational reports 2020, 2022, 2023",51.5
info_pct_new,fraction of informational reports 2024,21.6
r2023,reports in 2023,"36,210,368.0"
r2024,reports in 2024,"20,512,803.0"


In [2]:
show("<h2>Projected Growth</h2>")

f1 = r2024/i2024
g1 = f1 * growth

act_old = 1 - info_old
assert round(act_old, 5) == round(act_old_prime, 5), f"{act_old} v {act_old_prime}"
act_new = 1 - info_new
assert round(act_new, 5) == round(act_new_prime, 5), f"{act_old} v {act_old_prime}"
info_new_as_old = (act_old / act_new) * info_new

f2 = act_old + info_new_as_old
g2 = f2 * growth

projections = pd.DataFrame({
    "description": [
        "reports/incidents",
        "old actionable",
        "new actionable",
        "new informational as old",
        "old actionable + new informational as old",
        "f1 growth",
        "f2 growth",
        "years until deluge becomes too much again 1",
        "years until deluge becomes too much again 2",
    ],
    "value": [
        f1,
        act_old,
        act_new,
        info_new_as_old,
        f2,
        g1,
        g2,
        (40_000_000 - r2024) / g1,
        (40_000_000 - r2024) / g2,
    ]
}, index=[
    "f1",
    "act_old",
    "act_new",
    "info_new_as_old",
    "f2",
    "g1",
    "g2",
    "deluge1",
    "deluge2",
])

show("""
    <ul>
    <li>Baseline growth: Linear model 2014–2023
    <li>Factor 1: Reports / Incidents 2024
    <li>Factor 2: Historical Actionable Fraction + New Informational Fraction as Old
    </ul>
""")

show(projections, caption="Projections")

,description,value
f1,reports/incidents,0.702
act_old,old actionable,0.485
act_new,new actionable,0.784
info_new_as_old,new informational as old,0.133
f2,old actionable + new informational as old,0.618
g1,f1 growth,"2,759,559.552"
g2,f2 growth,"2,429,430.279"
deluge1,years until deluge becomes too much again 1,7.062
deluge2,years until deluge becomes too much again 2,8.021
